# Harmony Vector Flow Service (harmony-flow) Regression Tests

This notebook contains regression tests for the Harmony Vector Flow Service (`harmony-flow`). The service processes ocean current data from the OSCAR L4 NRT V2.0 collection, encoding the `u` and `v` surface current velocity components into a PNG image texture for use in particle flow visualizations in web maps.

**Note:** This service is deployed to production Harmony and can also be run against a local Harmony instance. Tests will be skipped in the SIT environment.

## Set the Harmony environment:

The cell below sets the `harmony_host_url` to one of the following valid values:

* Production: <https://harmony.earthdata.nasa.gov>
* UAT: <https://harmony.uat.earthdata.nasa.gov>
* SIT: <https://harmony.sit.earthdata.nasa.gov> *(harmony-flow not deployed)*
* Local: <http://localhost:3000>

The default value is for the UAT environment. When using this notebook there are two ways to use the non-default environment:

* Run this notebook in a local Jupyter notebook server and change the value of `harmony_host_url` in the cell below to the value for the environment you require from the above list.

* Use the `run_notebooks.sh` script, which requires you to declare an environment variable `HARMONY_HOST_URL`. Set that environment variable to the value above that corresponds to the environment you want to test. That environment variable will take precedence over the default value in the cell below.

In [ ]:
harmony_host_url = 'https://harmony.uat.earthdata.nasa.gov'

## Prerequisites

The dependencies for this notebook are listed in the [environment.yaml](./environment.yaml). To test or install locally, create the papermill environment used in the automated regression testing suite:

`conda env create -f ./environment.yaml && conda activate papermill-harmony-flow`

A `.netrc` file must also be located in the `test` directory of this repository.

### Import required packages:

In [ ]:
import tempfile
from pathlib import Path

import rasterio
from harmony import Client, Collection, Environment, Request

### Define helper functions:

In [ ]:
def print_error(error_string: str) -> None:
    """Print an error, with formatting for red text."""
    print(f'\033[91m{error_string}\033[0m')


def print_success(success_string: str) -> None:
    """Print a success message, with formatting for green text."""
    print(f'\033[92mSuccess: {success_string}\033[0m')


def print_warning(warning_string: str) -> None:
    """Print a warning, with formatting for yellow text."""
    print(f'\033[93m{warning_string}\033[0m')

### Set up environment dependent variables:

This service is deployed to production Harmony and is also available against a local Harmony instance. Tests in UAT and SIT environments will be skipped.

In [ ]:
host_environment = {
    'http://localhost:3000': Environment.LOCAL,
    'https://harmony.sit.earthdata.nasa.gov': Environment.SIT,
    'https://harmony.uat.earthdata.nasa.gov': Environment.UAT,
    'https://harmony.earthdata.nasa.gov': Environment.PROD,
}

harmony_environment = host_environment.get(harmony_host_url)

if harmony_environment is not None:
    harmony_client = Client(env=harmony_environment)

In [ ]:
oscar_collection = Collection(id='C1241042623-POCLOUD')
oscar_granule = 'G1242574718-POCLOUD'

## Test: OSCAR ocean current flow texture generation (`u`, `v`)

Submits an OSCAR L4 NRT V2.0 granule to the `harmony-flow` service requesting a PNG image texture encoding of ocean current data. The output PNG encodes the `u` component in the red channel and the `v` component in the green channel, each normalised to the `uint8` value range.

### Expected results:

* At least one PNG file is present in the job output.
* Each PNG contains exactly 3 bands (RGB).
* Each PNG has valid, non-zero spatial dimensions.

In [ ]:
harmony_flow_environments = {Environment.SIT, Environment.UAT, Environment.LOCAL}

if harmony_environment in harmony_flow_environments:
    oscar_request = Request(
        collection=oscar_collection,
        granule_id=oscar_granule,
        format='image/png',
        variables=['u', 'v'],
        labels=['harmony-flow-rtests'],
    )

    job_id = harmony_client.submit(oscar_request)
    harmony_client.wait_for_processing(job_id, show_progress=True)

    with tempfile.TemporaryDirectory() as temp_dir:
        downloaded_outputs = [
            file_future.result()
            for file_future in harmony_client.download_all(
                job_id, overwrite=True, directory=temp_dir
            )
        ]

        png_files = [
            Path(output)
            for output in downloaded_outputs
            if Path(output).suffix.lower() == '.png'
        ]

        assert png_files, 'No PNG files were returned by the harmony-flow service.'
        print_success('PNG output generated')

        for png_file in png_files:
            with rasterio.open(png_file) as dataset:
                assert (
                    dataset.count == 3
                ), f'Expected 3 bands in output PNG, got {dataset.count}.'
                assert dataset.width > 0 and dataset.height > 0, (
                    f'Output PNG has invalid dimensions: '
                    f'{dataset.width}x{dataset.height}.'
                )
                assert all(dtype == 'uint8' for dtype in dataset.dtypes), (
                    f'Expected uint8 band dtypes for u/v PNG encoding, '
                    f'got {dataset.dtypes}.'
                )
                print(
                    f'  Verified PNG: {dataset.width}x{dataset.height}, '
                    f'{dataset.count} bands, dtypes={dataset.dtypes}'
                )

        print_success('OSCAR u/v flow texture properties verified')
else:
    print_warning(
        f'Skipping harmony-flow test: Service is not deployed to '
        f'{harmony_environment.name if harmony_environment else "unknown"} '
        f'environment.'
    )